# FF++ Setup and Preprocessing (VM workbook)

**Cross-Generator Generalization in Deepfake Detection (IE7374)**

Run this top to bottom on the VM to go from nothing to a ready crops cache:
clone, install, verify environment, acquire FaceForensics++ c23, inventory,
preprocess. The crops cache then lives on the VM where training runs, so there
is no large upload to move around.

Three cells are marked **ADJUST** for your VM: the torch/CUDA install line, your
Kaggle credentials, and the data route (download vs. already staged on the VM).

## 1. Get the repo
Skip this cell if you already uploaded or cloned the repo and are inside it.

In [ ]:
!git clone https://github.com/domysticnom/Cross-Generator-Generalization-in-Deepfake-Detection.git
%cd Cross-Generator-Generalization-in-Deepfake-Detection

## 2. Install dependencies
Uses `{sys.executable} -m pip install --user` so packages land in the same Python this
kernel runs (mixing `pip` and `python` from different envs is why installs seem to
vanish), and `--user` works on a shared / read-only conda env.

If CUDA is not already working, installs **torch 2.4.1 + cu121**, which runs on any
driver at CUDA 12.1 or newer (covers the V100 and most clusters). We pin the version
and use the cu121 index ONLY: with an extra PyPI index, pip grabs a newer wheel built
for a CUDA your driver may be too old for.

**If this cell installs torch, RESTART THE KERNEL and re-run from here.** RTX 50-series
(Blackwell) instead needs cu128 and a newer torch.

In [ ]:
import importlib.util, sys
torch_ok = False
if importlib.util.find_spec("torch") is not None:
    import torch
    torch_ok = torch.cuda.is_available()
    print("torch", torch.__version__, "cuda:", torch_ok)
if not torch_ok:
    !{sys.executable} -m pip install --user torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
    print("installed torch 2.4.1+cu121: RESTART THE KERNEL, then re-run from step 2")
!{sys.executable} -m pip install --user -q -r requirements.txt
!{sys.executable} -m pip install --user -q mediapipe kaggle
import os
# --user installs the kaggle CLI into ~/.local/bin, often not on PATH; add it
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]

## 3. Verify the environment
Confirms Python, torch/CUDA, ffmpeg, and the core packages before spending compute.

In [ ]:
!python check_env.py

## 4. Kaggle credentials
Needed only for the download route in step 5. Skip if FF++ is already on the VM.
**ADJUST:** paste your token, or upload kaggle.json to ~/.kaggle/ and skip this cell.

In [ ]:
import json, pathlib
# ADJUST: fill in your Kaggle username/key (kaggle.com -> Account -> Create New API Token)
creds = {"username": "YOUR_USERNAME", "key": "YOUR_KEY"}
kdir = pathlib.Path.home() / ".kaggle"
kdir.mkdir(exist_ok=True)
(kdir / "kaggle.json").write_text(json.dumps(creds))
(kdir / "kaggle.json").chmod(0o600)
print("wrote", kdir / "kaggle.json")

## 5a. Download the FaceForensics++ c23 mirror
Uses the Kaggle Python API, which works even when the `kaggle` CLI is not on PATH
(common on shared VMs). `quiet=False` shows a progress bar. This is ~16.7 GB, then it
unzips (the unzip stage has no bar, so a cell sitting at ~100% is normal, not frozen).

Skip this cell if FF++ is already staged on the VM; set `--source-dir` in 5b instead.

In [ ]:
import kaggle
kaggle.api.authenticate()
kaggle.api.dataset_download_files("xdxd003/ff-c23", path="data/ffpp_kaggle",
                                  unzip=True, quiet=False)
print("download done")

## 5b. Normalize into data/raw/<method>/
Relocates the videos into the canonical layout. `--link-mode move` avoids doubling
disk. The script preserves native filename stems (the clip_id / source_id contract)
and ignores the mirror's extra folders (FaceShifter, DeepFakeDetection).

If FF++ was already on the VM, point `--source-dir` at that path instead.

In [ ]:
!python data/download_ffpp.py --route local --source-dir data/ffpp_kaggle --link-mode move

## 6. Verify the inventory
Must report 1000 clips per method (5000 total) and 0 corrupt, exit 0.
A non-zero exit means a partial or bad download, re-run route A (it is resumable).

In [ ]:
!python data/inventory_ffpp.py --raw data/raw --out data/manifests/ff_inventory.csv

## 7. Preprocess into the crops cache
Samples frames, detects and crops the face, resizes, and caches each crop as a
`.npy` under `data/processed/`, plus the `crops.parquet` manifest and a detection log.
Resumable via `--resume`. Knobs live in `configs/preprocess.yaml` (frames, size, seed).

This is the long step (roughly an hour for all 5000 clips on CPU).

In [ ]:
!python data/preprocess.py --raw data/raw --out data/processed \
  --manifest data/manifests/crops.parquet \
  --detect-log data/manifests/detection_log.csv \
  --resume

## 8. Sanity-check the cache
Confirm the manifest counts and eyeball one crop.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
m = pd.read_parquet("data/manifests/crops.parquet")
print("total crops:", len(m))
print(m.groupby("method").size())
row = m.iloc[0]
img = np.load(row["path"])
plt.imshow(img); plt.axis("off")
plt.title(f"{row['method']}  {row['crop_id']}  ({img.shape})")
plt.show()

## Next steps
The crops cache and `crops.parquet` now live on the VM. From here:

- build the leave-one-manipulation-out splits (`data/make_splits.py`),
- train EfficientNet / XceptionNet (`experiments/train.py --config ...`),
- evaluate into the transfer matrix (`experiments/evaluate.py`).

Because the cache is on the VM, teammates just re-run this notebook rather than
downloading a large archive. `docs/CACHE_FORMAT.md` documents the cache layout.